# 🎨 Notebook 1: What is a Backend-for-Frontend (BFF)?

A **Backend-for-Frontend (BFF)** is a small API gateway dedicated to **one type of client** — for example a mobile app, a web app, or a smart-TV app. Instead of one giant "universal" API that tries to serve every client equally, you build **several slim gateways**, each shaped for its audience.

> "One size fits all" rarely fits anyone well. A BFF fits *one* client perfectly.

### 🍽️ Everyday analogy

Imagine a restaurant:

- A **single buffet** (universal API) tries to feed toddlers, vegans, and bodybuilders with the same food laid out on the same table. Everyone takes something, but nobody gets the perfect plate.
- A **dedicated waiter per table** (BFF) asks each table what *they* want and brings back *only* that — plated the way they like it.

### 🌍 Real-world example: Netflix

Netflix famously uses BFFs. Their web browser, iPhone app, Android app, PS5 app, and smart-TV apps all have very different needs (screen size, input method, network speed, local storage). Each has its own BFF that calls the same underlying microservices (catalog, recommendations, billing) but shapes the response for *that* device.

### What you'll learn in this lab

1. **Why** one-API-for-everyone hurts at scale (this notebook).
2. **How** a BFF reshapes data per client and reduces bytes + latency (notebook 2).
3. **When** to use a BFF vs. an API Gateway, and common pitfalls (notebook 3).

## 🛠️ Setup

```bash
cd 05-microservices/bff
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.

This lab uses **only the Python standard library** — no servers to start, no Docker. We simulate HTTP calls with plain Python functions so you can focus on the pattern.

## 1. The problem — one API trying to please everyone

Different clients have **genuinely different needs**:

| Client       | Network        | Screen       | Power             | Typical payload wanted                        |
|--------------|----------------|--------------|-------------------|-----------------------------------------------|
| 📱 Mobile   | Slow, metered  | Small        | Battery-limited   | Tiny JSON, pre-formatted strings              |
| 💻 Web      | Fast           | Large        | Plenty            | Rich JSON, raw data for client-side rendering |
| 📺 Smart TV | Medium         | Huge         | Very limited CPU  | Pre-rendered text, big image URLs             |

If **one** API has to serve them all, it usually sends **everything** so that every client can pick what it needs. That means clients **over-fetch** (download bytes they throw away) or **under-fetch** (make many extra calls to get the last bit of data).

In [ ]:
# --- Downstream microservices (the actual backend) ---
# In a real system these would be separate services behind HTTP.
# We use plain functions so the notebook runs with zero setup.

def svc_user(uid: int) -> dict:
    return {
        'id': uid,
        'name': 'Ada Lovelace',
        'email': 'ada@example.com',
        'avatar_url': 'https://cdn.example.com/u/1/avatar-2048.png',
        'bio': 'Mathematician and writer. ' * 20,   # long bio
        'preferences': {'theme': 'dark', 'locale': 'en-GB', 'tz': 'Europe/London'},
        'security': {'mfa_enabled': True, 'last_login_ip': '10.0.0.1'},  # sensitive!
    }

def svc_orders(uid: int) -> list:
    return [
        {'id': i, 'total': round(9.99 * i, 2),
         'items': [{'sku': f'SKU-{i}-{j}', 'qty': 1, 'price': 9.99} for j in range(3)],
         'shipping_address': {'street': '1 Analytical Engine Rd', 'city': 'London'}}
        for i in range(1, 6)
    ]

def svc_recommendations(uid: int) -> list:
    return [{'sku': f'REC-{i}', 'title': f'Recommended item {i}',
             'score': round(0.99 - i*0.05, 2),
             'image_url': f'https://cdn.example.com/p/{i}/hero-4k.jpg'}
            for i in range(20)]


### ⚠️ Bad practice: the "universal" API

This endpoint returns **everything the app could possibly need**, regardless of who's asking. Every client — phone, web, TV — gets the same blob.

In [ ]:
import json

def universal_api(uid: int) -> dict:
    # Returns the full kitchen sink. Every client pays for every byte.
    return {
        'user': svc_user(uid),
        'orders': svc_orders(uid),
        'recommendations': svc_recommendations(uid),
    }

payload = universal_api(1)
print(f'universal API payload size: {len(json.dumps(payload)):,} bytes')
print(f'  - keys: {list(payload.keys())}')
print(f'  - first recommendation fields: {list(payload["recommendations"][0].keys())}')


Notice three problems with the universal response above:

1. **Over-fetching.** A mobile home screen might only show the user's name and an order count — yet it receives the full bio, every order line item, shipping addresses, and 20 recommendations.
2. **Security leakage.** Fields like `security.last_login_ip` or `shipping_address` may be irrelevant (and inappropriate) for some clients, but they leak out anyway.
3. **One team bottleneck.** Whoever owns this "universal" API must satisfy *every* frontend team. Every change needs coordination across product lines.

## 2. The BFF solution — one gateway per client

Each BFF is a thin server that:

1. Receives one request from **its** frontend.
2. Calls the downstream microservices it needs.
3. **Filters, reshapes, and aggregates** the result into exactly what its frontend wants.

Crucially, each BFF is **owned by the team that owns the frontend**. Mobile team → mobile BFF. Web team → web BFF. No more waiting on a shared "platform API" team for every UI tweak.

In [ ]:
def mobile_bff(uid: int) -> dict:
    """Tiny payload for mobile home screen - just the headline numbers."""
    user = svc_user(uid)
    orders = svc_orders(uid)
    return {
        'name': user['name'],
        'order_count': len(orders),
        'theme': user['preferences']['theme'],
    }

def web_bff(uid: int) -> dict:
    """Rich payload for a web dashboard - full data for client-side rendering."""
    return {
        'user': {k: v for k, v in svc_user(uid).items() if k != 'security'},
        'orders': svc_orders(uid),
        'recommendations': svc_recommendations(uid)[:10],
    }

def tv_bff(uid: int) -> dict:
    """Smart TV: pre-rendered greeting + 5 big recs with 4k images."""
    user = svc_user(uid)
    recs = svc_recommendations(uid)[:5]
    return {
        'greeting': f"Welcome back, {user['name'].split()[0]}!",
        'featured': [{'title': r['title'], 'image': r['image_url']} for r in recs],
    }

import json
for name, fn in [('mobile', mobile_bff), ('web', web_bff), ('tv', tv_bff)]:
    payload = fn(1)
    print(f'{name:7s} BFF: {len(json.dumps(payload)):>6,} bytes  - fields: {list(payload.keys())}')


### 🎯 Why each BFF is better than the universal API

- **Mobile** drops from thousands of bytes to ~90. Fewer packets, faster screen, less battery.
- **Web** gets rich data but *without* security fields — the BFF acts as a safety filter.
- **Smart TV** receives a pre-rendered greeting string and big image URLs tailored for its large screen. The TV code doesn't have to do string formatting or choose between image sizes.

Each BFF:

- **Owns its shape**: the mobile team can change the mobile payload without asking anyone.
- **Aggregates for its client**: one request in, one tailored response out (no chatty `N+1` from the phone).
- **Protects downstream services**: services return their natural data; the BFF handles filtering, translation, and formatting per client.

## 3. BFF vs. API Gateway — quick preview

People often confuse these. A short version:

| Concern            | **API Gateway**                 | **BFF**                                  |
|--------------------|---------------------------------|------------------------------------------|
| Scope              | All clients share it            | **One** client type only                 |
| Owner              | Platform / infra team           | The frontend team that uses it           |
| Typical logic      | Auth, rate-limit, routing, TLS  | Aggregation, shaping, client-specific UX |
| "Opinion" about UI | None — generic                  | Strong — designed for one UI            |

You often see both together: a single shared API Gateway does auth/rate-limiting, and behind it sits one BFF per client. We'll dig deeper in notebook 3.

## 🧠 Key takeaways

- A **BFF is a small gateway per client type**, owned by that client's team.
- It exists because **clients genuinely differ** (network, screen, power, UX).
- A BFF **aggregates, filters, and reshapes** downstream data so each client gets exactly what it needs.
- Benefits: **smaller payloads**, **faster screens**, **clearer ownership**, **better security boundary**.
- Cost: **more gateways to operate**. Only add a BFF when client needs truly diverge.

👉 Next: [`02_worked_example.ipynb`](./02_worked_example.ipynb) — measure the latency and byte savings with real numbers, and add **parallel fan-out** and **graceful degradation**.